In [ ]:
import random
import os
import numpy as np
import time
import torch
import torch.nn as nn 
import pandas as pd 
import argparse
import torch.optim as optim
import torch.utils.data as data
from tensorboardX import SummaryWriter

In [ ]:
DATA_URL = "http://files.grouplens.org/datasets/movielens/ml-100k/u.data"

# MAIN_PATH = '/home/yyeon/KeepGo/Neural_Collaborative_Filtering/'

# DATA_PATH = MAIN_PATH + 'data/ml-1m/ratings.dat'
# MODEL_PATH = MAIN_PATH + 'models/'
MAIN_PATH = r'D:\JupyterCode\NCF'
DATA_PATH = MAIN_PATH + '\\data\\ml-1m\\ratings.dat'
MODEL_PATH = MAIN_PATH + '\\models\\'

MODEL = 'ml-1m_Neu_MF'

载入数据：

In [ ]:
# load data
ml_1m = pd.read_csv(
    DATA_PATH, 
    sep="::", 
    names = ['user_id', 'item_id', 'rating', 'timestamp'], 
    engine='python')

参数设置：

In [ ]:
#设置伪随机数，方便实验结果复现
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 即使没有 CUDA，也可以设置，不会产生影响
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # 在没有 CUDA 的情况下通常将其设置为 False

In [ ]:
parser = argparse.ArgumentParser() 
parser.add_argument("--seed", type=int, default=42,help="Seed")
parser.add_argument("--lr", type=float, default=0.001, help="learning rate")
parser.add_argument("--dropout",type=float,default=0.2, help="dropout rate")
parser.add_argument("--batch_size", type=int, default=256,help="batch size for training")
parser.add_argument("--epochs", type=int,default=10,  help="training epoches")
parser.add_argument("--top_k", type=int, default=5, help="compute metrics@top_k")
parser.add_argument("--factor_num", type=int,default=32, help="predictive factors numbers in the model")
parser.add_argument("--layers",nargs='+', default=[64,32,16,8], help="MLP layers. Note that the first layer is the concatenation of user and item embeddings. So layers[0]/2 is the embedding size.")
parser.add_argument("--num_ng", type=int,default=4, help="Number of negative samples for training set")
parser.add_argument("--num_ng_test", type=int,default=100, help="Number of negative samples for test set")
parser.add_argument("--out", default=True,help="save model or not")

# set device and parameters
# args = parser.parse_args()有问题解决看： https://blog.csdn.net/wmq104/article/details/123534597
args=parser.parse_known_args()[0]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter()#将各种信息写入 TensorBoard 日志文件，以便在 TensorBoard 界面中查看和分析实验结果。

# seed for Reproducibility
# util.seed_everything(args.seed)
seed_everything(args.seed)

下面是 argparse.ArgumentParser() 函数的详细解释：
parser = argparse.ArgumentParser(
    description='Program Description',
    epilog='Program Epilog'
)
description（可选参数）：用于指定程序的简短描述或概述，通常会在用户运行 --help 或 -h 选项时显示在帮助文档中。
epilog（可选参数）：用于指定程序的结尾说明，通常会在用户运行 --help 或 -h 选项时显示在帮助文档中。
argparse.ArgumentParser() 创建了一个 ArgumentParser 对象，你可以使用它来定义和解析命令行参数。

可以使用 add_argument() 方法向 ArgumentParser 对象添加命令行参数的定义。
--lr：是命令行参数的名称，以 -- 开头表示这是一个长选项。
type=float：指定参数的类型，这里是浮点数。
default=0.001：指定参数的默认值。
help：提供了关于参数的帮助文本，通常在用户运行 --help 或 -h 选项时显示。
添加完所有参数的定义后，你可以使用 parse_args() 方法来解析命令行参数，并将它们存储在一个 Namespace 对象中。
你可以通过 args 对象来访问命令行参数的值，例如 args.lr 来获取学习率的值。

处理数据：

In [ ]:
"""
这个自定义数据集类的主要目的是将原始数据组织成适用于 PyTorch 模型训练和测试的形式。
通过实现这些方法，可以方便地使用 PyTorch 提供的数据加载器来加载和处理数据，以供深度学习模型使用。
"""
class Rating_Datset(torch.utils.data.Dataset):
    def __init__(self, user_list, item_list, rating_list):
        super(Rating_Datset, self).__init__()
        self.user_list = user_list
        self.item_list = item_list
        self.rating_list = rating_list

    def __len__(self):
        return len(self.user_list)

    def __getitem__(self, idx):
        user = self.user_list[idx]
        item = self.item_list[idx]
        rating = self.rating_list[idx]

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(item, dtype=torch.long),
            torch.tensor(rating, dtype=torch.float)
            )

In [ ]:
class NCF_Data(object):
    """
    Construct Dataset for NCF
    """
    def __init__(self, args, ratings):
        self.ratings = ratings
        self.num_ng = args.num_ng
        self.num_ng_test = args.num_ng_test
        self.batch_size = args.batch_size

        self.preprocess_ratings = self._reindex(self.ratings)

        self.user_pool = set(self.ratings['user_id'].unique())
        self.item_pool = set(self.ratings['item_id'].unique())

        self.train_ratings, self.test_ratings = self._leave_one_out(self.preprocess_ratings)
        self.negatives = self._negative_sampling(self.preprocess_ratings)
        random.seed(args.seed)

        
    """
    本函数作用：
    1.用整数重新索引
    2.将有无交互映射为0，1

    """   
    def _reindex(self, ratings):
        """
        Process dataset to reindex userID and itemID, also set rating as binary feedback
        """
        user_list = list(ratings['user_id'].drop_duplicates())
        user2id = {w: i for i, w in enumerate(user_list)}
        item_list = list(ratings['item_id'].drop_duplicates())
        item2id = {w: i for i, w in enumerate(item_list)}
#  enumerate(item_list) 是一个迭代器，用于遍历 item_list 列表中的元素。
# 在每次迭代中，enumerate 返回一个元组，其中第一个元素 i 是元素的索引（从0开始），第二个元素 w 是元素的值（即物品ID）。
        ratings['user_id'] = ratings['user_id'].apply(lambda x: user2id[x])
        ratings['item_id'] = ratings['item_id'].apply(lambda x: item2id[x])
# .apply() 方法的作用，ratings['item_id'] 列中的每个原始物品ID都被替换为了相应的整数ID，完成了物品ID的重新索引
        ratings['rating'] = ratings['rating'].apply(lambda x: float(x > 0))
# .apply() 方法的作用，ratings['rating'] 列中的每个原始评分值都被替换为了相应的二进制反馈值。
        return ratings
                                             
                                                
    """
    这种 "leave-one-out" 评估协议的核心思想是将每个用户的最新交互数据作为测试集，其余数据作为训练集，以评估模型的性能。
    这有助于模拟实际应用场景中的推荐任务，其中用户的历史行为用于训练模型，最新行为用于测试模型。

    """
    def _leave_one_out(self, ratings):
        """
        leave-one-out evaluation protocol in paper https://www.comp.nus.edu.sg/~xiangnan/papers/ncf.pdf
        """
        ratings['rank_latest'] = ratings.groupby(['user_id'])['timestamp'].rank(method='first', ascending=False)
        test = ratings.loc[ratings['rank_latest'] == 1]
        train = ratings.loc[ratings['rank_latest'] > 1]
        assert train['user_id'].nunique()==test['user_id'].nunique(), 'Not Match Train User with Test User'
        return train[['user_id', 'item_id', 'rating']], test[['user_id', 'item_id', 'rating']]


    def _negative_sampling(self, ratings):
        interact_status = (#一个DataFrame
            ratings.groupby('user_id')['item_id']
            .apply(set)#.apply(set)：然后，对于每个用户，它将其交互过的物品ID放入一个集合（set）中，以去除重复的物品。
            .reset_index()#它重置索引，以便得到一个包含用户ID和其交互过的物品ID集合的 DataFrame。
            .rename(columns={'item_id': 'interacted_items'}))
        interact_status['negative_items'] = interact_status['interacted_items'].apply(lambda x: self.item_pool - x)
#通过从整体物品池 self.item_pool 中减去每个用户的交互物品集合得到的。这意味着 negative_items 包含了每个用户未曾交互的物品。
        interact_status['negative_samples'] = interact_status['negative_items'].apply(lambda x: random.sample(x, self.num_ng_test))
#negative_samples 的列，其中包含了每个用户的负样本样本。这是通过从 negative_items 中随机抽样 self.num_ng_test 个物品得到的
        return interact_status[['user_id', 'negative_items', 'negative_samples']]

    def get_train_instance(self):
        users, items, ratings = [], [], []
        train_ratings = pd.merge(self.train_ratings, self.negatives[['user_id', 'negative_items']], on='user_id')
# 将训练评分数据集 self.train_ratings 与负样本数据 self.negatives[['user_id', 'negative_items']] 进行合并，以获得包含正样本和负样本的数据                                              
        train_ratings['negatives'] = train_ratings['negative_items'].apply(lambda x: random.sample(x, self.num_ng))
# 创建了一个新的列 negatives，其中包含了每个用户的负样本物品列表，这些负样本物品是从 negative_items 中随机抽样的，数量由 self.num_ng 决定                                              
        for row in train_ratings.itertuples():
            users.append(int(row.user_id))
            items.append(int(row.item_id))
            ratings.append(float(row.rating))
            for i in range(self.num_ng):#添加负样本
                users.append(int(row.user_id))
                items.append(int(row.negatives[i]))
                ratings.append(float(0))  # negative samples get 0 rating
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)#num_workers=4改成0，4会发生broken pipe

    def get_test_instance(self):
        users, items, ratings = [], [], []
        test_ratings = pd.merge(self.test_ratings, self.negatives[['user_id', 'negative_samples']], on='user_id')
        for row in test_ratings.itertuples():
            users.append(int(row.user_id))
            items.append(int(row.item_id))
            ratings.append(float(row.rating))
            for i in getattr(row, 'negative_samples'):
                users.append(int(row.user_id))
                items.append(int(i))
                ratings.append(float(0))
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.num_ng_test+1, shuffle=False, num_workers=0)#num_workers=4改成0


In [ ]:
# construct the train and test datasets
data = NCF_Data(args, ml_1m)
print(data)
train_loader =data.get_train_instance()
test_loader =data.get_test_instance()

模型架构

In [ ]:
class Generalized_Matrix_Factorization(nn.Module):
    def __init__(self, args, num_users, num_items):#args包含模型的超参数
        super(Generalized_Matrix_Factorization, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.factor_num = args.factor_num#表示嵌入向量的维度

        self.embedding_user = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num)
        self.embedding_item = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num)

        self.affine_output = nn.Linear(in_features=self.factor_num, out_features=1)#全连接层
        self.logistic = nn.Sigmoid()

    def forward(self, user_indices, item_indices):#前向传播函数
        user_embedding = self.embedding_user(user_indices)
        item_embedding = self.embedding_item(item_indices)
        element_product = torch.mul(user_embedding, item_embedding)
        logits = self.affine_output(element_product)#元素积
        rating = self.logistic(logits)
        return rating

    def init_weight(self):
        pass

class Multi_Layer_Perceptron(nn.Module):
    def __init__(self, args, num_users, num_items):
        super(Multi_Layer_Perceptron, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.factor_num = args.factor_num
        self.layers = args.layers

        self.embedding_user = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num)
        self.embedding_item = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num)

        self.fc_layers = nn.ModuleList()#创建了空列表用于存储多个全连接层
        for idx, (in_size, out_size) in enumerate(zip(self.layers[:-1], self.layers[1:])):#切片的语法
# 这是一个循环，它同时迭代遍历 self.layers[:-1] 和 self.layers[1:] 中的元素对，将它们分配给变量 idx（索引）、in_size（输入维度）和 out_size（输出维度）。
            self.fc_layers.append(nn.Linear(in_size, out_size))
# append 方法将这个全连接层对象添加到 self.fc_layers 中，将其存储起来以备后续使用
        self.affine_output = nn.Linear(in_features=self.layers[-1], out_features=1)
        self.logistic = nn.Sigmoid()

    def forward(self, user_indices, item_indices):
        user_embedding = self.embedding_user(user_indices)
        item_embedding = self.embedding_item(item_indices)
        vector = torch.cat([user_embedding, item_embedding], dim=-1)  # the concat latent vector
        for idx, _ in enumerate(range(len(self.fc_layers))):
            vector = self.fc_layers[idx](vector)
            vector = nn.ReLU()(vector)
            # vector = nn.BatchNorm1d()(vector)
            # vector = nn.Dropout(p=0.5)(vector)
        logits = self.affine_output(vector)
        rating = self.logistic(logits)
        return rating

    def init_weight(self):
        pass

enumerate(iterable): enumerate 是一个内置函数，用于将可迭代对象（例如列表或元组）的元素与它们的索引一起迭代。
在这个代码中，enumerate 用于遍历 self.layers[:-1] 列表中的每个元素（隐藏层的输入维度），并为每个元素生成一个索引 idx。

zip(iterable1, iterable2): zip 也是一个内置函数，用于将两个或多个可迭代对象的元素逐一配对。
在这个代码中，zip 用于将 self.layers[:-1]（输入维度列表）和 self.layers[1:]（输出维度列表）两个可迭代对象的元素一一配对，这样就可以同时获取每个隐藏层的输入维度 in_size 和输出维度 out_size。

for idx, (in_size, out_size) in enumerate(zip(self.layers[:-1], self.layers[1:])):: 这是一个循环，它同时迭代遍历 self.layers[:-1] 和 self.layers[1:] 中的元素对，并将它们分配给变量 idx（索引）、in_size（输入维度）和 out_size（输出维度）。

self.fc_layers.append(nn.Linear(in_size, out_size)): 在循环内部，针对每个隐藏层，nn.Linear(in_size, out_size) 创建了一个全连接层对象，其中 in_size 是输入维度，out_size 是输出维度。然后，append 方法将这个全连接层对象添加到 self.fc_layers 中，将其存储起来以备后续使用。

总结：这段代码的目的是创建多个全连接层，每个全连接层具有不同的输入维度和输出维度，这些维度通过遍历 self.layers[:-1] 和 self.layers[1:] 的元素对来确定。这使得模型可以包含多个隐藏层，每个隐藏层的大小可以根据 self.layers 列表中的定义进行配置。

In [ ]:
class NeuMF(nn.Module):
    def __init__(self, args, num_users, num_items):
        super(NeuMF, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.factor_num_mf = args.factor_num#MF部分的潜在因子数量32
        self.factor_num_mlp =  int(args.layers[0]/2)#MLP部分的潜在因子数量32
        self.layers = args.layers #MLP各层大小
        self.dropout = args.dropout #正则化dropout rating

        self.embedding_user_mlp = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num_mlp)
        self.embedding_item_mlp = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num_mlp)

        self.embedding_user_mf = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num_mf)
        self.embedding_item_mf = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num_mf)

        self.fc_layers = nn.ModuleList()
#       由多个全连接层组成的列表，每个全连接层后跟一个ReLU激活函数
        for idx, (in_size, out_size) in enumerate(zip(args.layers[:-1], args.layers[1:])):
            self.fc_layers.append(torch.nn.Linear(in_size, out_size))
            self.fc_layers.append(nn.ReLU())

        self.affine_output = nn.Linear(in_features=args.layers[-1] + self.factor_num_mf, out_features=1)
        self.logistic = nn.Sigmoid()
        self.init_weight()

    """
    init_weight 方法通过不同的初始化策略对模型的权重进行初始化，
    以确保模型在训练过程中能够更好地收敛，并且不会出现梯度问题
    """   
    def init_weight(self):
        nn.init.normal_(self.embedding_user_mlp.weight, std=0.01)
        nn.init.normal_(self.embedding_item_mlp.weight, std=0.01)
        nn.init.normal_(self.embedding_user_mf.weight, std=0.01)
        nn.init.normal_(self.embedding_item_mf.weight, std=0.01)
        
        for m in self.fc_layers:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                
        nn.init.xavier_uniform_(self.affine_output.weight)

        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                m.bias.data.zero_()#将当前子模块的偏差（如果存在）初始化为零

    def forward(self, user_indices, item_indices):
        user_embedding_mlp = self.embedding_user_mlp(user_indices)
        item_embedding_mlp = self.embedding_item_mlp(item_indices)

        user_embedding_mf = self.embedding_user_mf(user_indices)
        item_embedding_mf = self.embedding_item_mf(item_indices)

        mlp_vector = torch.cat([user_embedding_mlp, item_embedding_mlp], dim=-1)  # the concat latent vector
        mf_vector =torch.mul(user_embedding_mf, item_embedding_mf)

        for idx, _ in enumerate(range(len(self.fc_layers))):
#实现了 MLP 的前向传播过程，它将输入数据 mlp_vector 通过 MLP 中的每一层逐层传递，最终得到 MLP 的输出
            mlp_vector = self.fc_layers[idx](mlp_vector)

        vector = torch.cat([mlp_vector, mf_vector], dim=-1)
        logits = self.affine_output(vector)
        rating = self.logistic(logits)
        return rating.squeeze()
        #return rating

损失函数：

In [ ]:
loss_function = nn.BCELoss()

评估：

In [ ]:
def hit(ng_item, pred_items):
    if ng_item in pred_items:
        return 1
    return 0

def ndcg(ng_item, pred_items):
    if ng_item in pred_items:
        index = pred_items.index(ng_item)
        return np.reciprocal(np.log2(index+2))
    return 0

def metrics(model, test_loader, top_k, device):
    HR, NDCG = [], []

    for user, item, label in test_loader:
        user = user.to(device)
        item = item.to(device)

        predictions = model(user, item)
        _, indices = torch.topk(predictions, top_k)
        recommends = torch.take(
                item, indices).cpu().numpy().tolist()

        ng_item = item[0].item() # leave one-out evaluation has only one item per user
        HR.append(hit(ng_item, recommends))
        NDCG.append(ndcg(ng_item, recommends))

    return np.mean(HR), np.mean(NDCG)

In [ ]:
# set the num_users, items
num_users = ml_1m['user_id'].nunique()+1
num_items = ml_1m['item_id'].nunique()+1
# ml_1m['user_id'].nunique()：这部分代码首先从 ml_1m 数据集中选择列 user_id，
# 然后使用 nunique() 方法来计算唯一用户的数量。每个用户通常有一个唯一的用户ID，所以这个操作返回数据集中不同用户的数量。
# +1：通常在计算用户数量和物品数量时，会将一个额外的数值加上去，以便考虑到可能的用户或物品ID从0开始的情况。

# set model and optimizer
model =NeuMF(args, num_users, num_items)
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=args.lr)

# train, evaluation
best_hr = 0
for epoch in range(1, args.epochs+1):
    model.train() # Enable dropout (if have).
# model.train() 是 PyTorch 中的一个方法，用于将模型设置为训练模式。让我解释一下这个方法的作用和用途：
# 在深度学习中，训练模型和评估模型是两个不同的模式。模型在训练模式下时，它会执行以下操作：
# 启用 dropout 和批量归一化（如果模型中使用了这些操作）。这意味着在训练模式下，模型会在前向传播过程中应用 dropout 层，
# 以防止过拟合，并且会更新批量归一化的统计信息。
# 模型会保留每个中间层的梯度，以便进行反向传播和参数更新。
# 在训练模式下，通常会使用不同的数据批次进行多次迭代，以便更新模型参数来最小化损失函数。
    start_time = time.time()

    for user, item, label in train_loader:
        user = user.to(device)
        item = item.to(device)
        label = label.to(device)

        optimizer.zero_grad()#清零梯度
        prediction = model(user, item)
# 在 PyTorch 中，当你定义了一个继承自 nn.Module 的模型类时，模型的前向传播逻辑会被实现在 forward 方法中。
# 当你调用模型实例（例如，model(user, item)）时，PyTorch 会自动调用模型的 forward 方法来执行前向传播操作，
# 因此你不需要显式调用 model.forward()。
# 简而言之，model(user, item) 与 model.forward(user, item) 是等效的。在代码中，一般会使用 model(user, item) 来执行前向传播，
# 因为这更简洁和方便。 PyTorch 的设计让模型的使用更接近自然语言，提高了代码的可读性。
        
        loss = loss_function(prediction, label)
#         度量模型的预测与真实标签之间的差异
        loss.backward()
#         反向传播的过程，它用于计算如何更新模型的参数以降低损失
        optimizer.step()
#     优化器根据计算得到的梯度来更新模型的权重和偏置，以降低损失函数的值
        writer.add_scalar('loss/Train_loss', loss.item(), epoch)
#     将训练损失值记录到 TensorBoard 日志中，以便后续可视化和分析。这有助于监控模型的训练过程，了解损失如何随着时间（epoch）的推移而变化。
    

    model.eval()
#     model.eval() 是 PyTorch 中的一个方法，用于将神经网络模型切换到评估（inference）模式。
# 当调用这个方法时，模型会进入一个状态，该状态适用于在验证集或测试集上进行推断操作，而不是训练。
# 主要的变化包括以下几点：
# Dropout 和 Batch Normalization 的影响： 在训练时，神经网络通常使用 dropout 层和批归一化（Batch Normalization）层来防止过拟合和加速训练。但在评估模型性能时，我们不希望随机丢弃神经元或改变批次统计信息。因此，model.eval() 会将 dropout 层和批归一化层的行为设置为不使用（即固定参数），以确保模型在评估时的输出是确定性的。
# 梯度计算： 在评估模式下，PyTorch 不会计算梯度信息，因为我们不需要在推断时进行参数更新。这节省了计算资源，并且可以提高模型的计算速度。
    
    HR, NDCG = metrics(model, test_loader, args.top_k, device)
    writer.add_scalar('Perfomance/HR@10', HR, epoch)
    writer.add_scalar('Perfomance/NDCG@10', NDCG, epoch)

    elapsed_time = time.time() - start_time
    print("The time elapse of epoch {:03d}".format(epoch) + " is: " + 
            time.strftime("%H: %M: %S", time.gmtime(elapsed_time)))
    print("HR: {:.5f}\tNDCG: {:.5f}".format(np.mean(HR), np.mean(NDCG)))

    if HR > best_hr:
        best_hr, best_ndcg, best_epoch = HR, NDCG, epoch
        if args.out:
            if not os.path.exists(MODEL_PATH):#删去了config.
                os.mkdir(MODEL_PATH)
            torch.save(model, 
                '{}{}.pth'.format(MODEL_PATH, MODEL))

writer.close()
print("End. Best epoch {:03d}: HR = {:.5f}, NDCG = {:.5f}".format(best_epoch, best_hr, best_ndcg))

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# 每个 epoch 的 HR 和 NDCG 值
epochs = list(range(1, 31))
hr_values = [0.629, 0.662, 0.668, 0.667, 0.669, 0.674, 0.672, 0.670, 0.668, 0.664, 0.663, 0.659, 0.654, 0.657, 0.658, 0.659, 0.652, 0.649, 0.656, 0.646, 0.651, 0.646, 0.646, 0.644, 0.643, 0.643, 0.637, 0.637, 0.637, 0.637]
ndcg_values = [0.359, 0.389, 0.398, 0.396, 0.398, 0.399, 0.396, 0.396, 0.396, 0.393, 0.392, 0.390, 0.388, 0.387, 0.386, 0.388, 0.385, 0.382, 0.384, 0.379, 0.383, 0.380, 0.379, 0.376, 0.376, 0.376, 0.374, 0.372, 0.373, 0.374]

# 创建折线图
plt.figure(figsize=(10, 5))
plt.plot(epochs, hr_values, label='HR', marker='o')
plt.plot(epochs, ndcg_values, label='NDCG', marker='s')
plt.title('HR and NDCG over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Value')
plt.legend()
plt.grid(True)

# 保存图表
# plt.savefig('hr_ndcg_plot.png')

# 显示图表
plt.show()